In [42]:
import os
#from pathlib import Path
#import re
import pandas as pd
import numpy as np
from PIL import Image
from skimage.color import rgb2gray
from skimage.draw import disk
from matplotlib.patches import Circle
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
import tifffile

In [2]:
def MFI_foci(
        image_path,
        df,
        px_size_ts_x = 11.6,
        px_size_ts_y = 11.6,
        px_size_x = 57.5,
        px_size_y = 58.7,
        x_col="x [nm]",
        y_col="y [nm]",
        sigma_col="sigma [nm]"
    ):

        # Open image
        image = Image.open(image_path).convert("RGB")

        # Convert image to grayscale
        gray = rgb2gray(image)
        H, W = gray.shape # number of pixels

        # Scaling factors
        sx = px_size_ts_x/px_size_x
        sy = px_size_ts_y/px_size_y
        ssigma = np.mean([px_size_ts_x, px_size_ts_y]) / np.mean([px_size_x, px_size_y])

        # Storage lists
        x_list = []
        y_list = []
        sigma_list = []
        mean_list = []

        for _, row in df.iterrows():
            x_nm = row[x_col]
            y_nm = row[y_col]
            sigma_nm = row[sigma_col]

            # original pixels → current image pixels
            x_px = int(round(sx * x_nm / px_size_ts_x))
            y_px = int(round(sy * y_nm / px_size_ts_y))
            sigma_px = max(1, int(round(ssigma * sigma_nm / np.mean([px_size_ts_x, px_size_ts_y])))) # minimal possible value is 1 pixel!

            # Build circular mask (clipped automatically)
            rr, cc = disk((y_px, x_px), sigma_px, shape=(H, W))
            mask = np.zeros((H, W), dtype=bool)
            mask[rr, cc] = True
            disk((y_px, x_px), sigma_px, shape=(H, W))

            # Compute mean intensity
            if mask.sum() > 0:
                mean_intensity = gray[mask].mean()
            else:
                mean_intensity = np.nan

            x_list.append(x_px)
            y_list.append(y_px)
            sigma_list.append(sigma_px)
            mean_list.append(mean_intensity)

        # Return modified copy
        df_out = df.copy()
        df_out["x_px"] = x_list
        df_out["y_px"] = y_list
        df_out["sigma_px"] = sigma_list
        df_out["mean_intensity"] = mean_list

        return df_out

In [49]:
#image_path = "/mnt/c/users/elopatukhin/Desktop/Miscroscopy/160226_U2OS_fixed/MP_WT_0.3/C2_MP_U2OS_fixed_siORC1_WT0.3_001.nd2_(series_01).tif"
image_path = "/mnt/c/users/elopatukhin/Desktop/Miscroscopy/160226_U2OS_fixed/MP_WT_0.3/foci/C2_MP_U2OS_fixed_siORC1_WT0.3_001.nd2_(series_01)_ROI_0234-0314_foci.png"

image = Image.open(image_path) # open image
matrix = np.array(image) # convert image to matrix
matrix = np.array(image, dtype=np.float32) # convert values to float
#H, W = matrix.shape # get number of pixels (512*512 for 16-bit image)
px_size_nm = 58.739

In [39]:
f = "/mnt/c/users/elopatukhin/Desktop/Miscroscopy/160226_U2OS_fixed/MP_WT_0.3/foci/C2_MP_U2OS_fixed_siORC1_WT0.3_001.nd2_(series_01)_ROI_0234-0314_foci.csv"
df = pd.read_csv(f)
px_size_ts_x = 1
px_size_ts_y = 1
px_size_x = 58.7
px_size_y = 58.7

In [50]:
with tifffile.TiffFile(image_path) as tif:
    page = tif.pages[0]

    xres = page.tags["XResolution"].value
    yres = page.tags["YResolution"].value

    x_pixels_per_micron = xres[0] / xres[1]
    y_pixels_per_micron = yres[0] / yres[1]

    print(x_pixels_per_micron, y_pixels_per_micron)

TiffFileError: not a TIFF file: header=b'\x89PNG'